In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import os
import shap
import matplotlib.pyplot as plt
import spacy
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
nlp = spacy.load("en_core_web_sm")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [5]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered.csv')

# EDA

In [6]:
df_female = df[df['label'] == 0]
df_male = df[df['label'] == 1]

In [7]:
def extract_pos_words(texts, pos_tag):
    words = []
    for doc in nlp.pipe(texts, disable=["ner", "parser"]):  # Faster
        for token in doc:
            if token.pos_ == pos_tag and not token.is_stop and token.is_alpha:
                words.append(token.lemma_.lower())
    return words

# Adjectives

In [8]:
# Extract POS-specific words using full context
female_adjs = extract_pos_words(df_female["full_text"], "ADJ")
male_adjs   = extract_pos_words(df_male["full_text"], "ADJ")

# Join into single strings
female_doc = " ".join(female_adjs)
male_doc   = " ".join(male_adjs)

documents = [female_doc, male_doc]


In [9]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    tfidf_matrix.T.toarray(),
    index=vectorizer.get_feature_names_out(),
    columns=["female", "male"]
)
tfidf_df["diff"] = tfidf_df["male"] - tfidf_df["female"]

In [10]:
# Most male-associated adjectives
top_male_adjs = tfidf_df.sort_values("diff", ascending=False).head(10)

# Most female-associated adjectives
top_female_adjs = tfidf_df.sort_values("diff", ascending=True).head(10)

In [11]:
top_male_adjs

,female,male,diff
calm,0.021404,0.031890,0.010486
young,0.030375,0.040406,0.010031
medical,0.609714,0.619496,0.009782
good,0.144768,0.154044,0.009275
professional,0.064924,0.074169,0.009246
ethic,0.055420,0.064632,0.009211
long,0.057019,0.065654,0.008635
internal,0.056397,0.064164,0.007766
appropriate,0.033128,0.040831,0.007703
respectful,0.016786,0.024482,0.007696


In [12]:
top_female_adjs

,female,male,diff
identifi,0.404463,0.382980,-0.021483
outstanding,0.126650,0.111211,-0.015439
clinical,0.320622,0.308854,-0.011768
pediatric,0.043608,0.033721,-0.009887
public,0.013766,0.007025,-0.006741
global,0.011546,0.005663,-0.005883
identifier,0.053733,0.048580,-0.005153
numerous,0.028154,0.023204,-0.004950
new,0.057019,0.052242,-0.004777
social,0.017408,0.012731,-0.004677


# Verbs

In [13]:
female_verbs = extract_pos_words(df_female["full_text"], "VERB")
male_verbs   = extract_pos_words(df_male["full_text"], "VERB")

female_doc = " ".join(female_verbs)
male_doc   = " ".join(male_verbs)

documents = [female_doc, male_doc]


In [14]:
# Run TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.T.toarray(),
    index=vectorizer.get_feature_names_out(),
    columns=["female", "male"]
)

# Compute male-female difference
tfidf_df["diff"] = tfidf_df["male"] - tfidf_df["female"]

# Show top verbs
top_male_verbs = tfidf_df.sort_values("diff", ascending=False).head(10)
top_female_verbs = tfidf_df.sort_values("diff", ascending=True).head(10)


In [15]:
top_male_verbs

,female,male,diff
show,0.105123,0.121443,0.016321
like,0.046116,0.058660,0.012544
feel,0.104389,0.114893,0.010504
believe,0.092022,0.100338,0.008316
learn,0.221565,0.228380,0.006815
display,0.034482,0.041047,0.006565
know,0.153859,0.160210,0.006351
spend,0.071375,0.077291,0.005916
supervise,0.021800,0.026783,0.004982
build,0.016140,0.020863,0.004723


In [16]:
top_female_verbs

,female,male,diff
take,0.133840,0.123918,-0.009922
complete,0.102293,0.093642,-0.008651
stand,0.038150,0.030276,-0.007874
support,0.052719,0.044977,-0.007741
organize,0.035530,0.027947,-0.007583
present,0.078292,0.071420,-0.006872
excel,0.065715,0.059096,-0.006619
match,0.034377,0.028432,-0.005945
care,0.054291,0.048568,-0.005723
recruit,0.029032,0.023483,-0.005549


# Nouns

In [17]:
# Extract nouns from female and male letters
female_nouns = extract_pos_words(df_female["full_text"], "NOUN")
male_nouns   = extract_pos_words(df_male["full_text"], "NOUN")

# Combine into pseudo-documents
female_doc = " ".join(female_nouns)
male_doc   = " ".join(male_nouns)

documents = [female_doc, male_doc]


In [18]:
# Run TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.T.toarray(),
    index=vectorizer.get_feature_names_out(),
    columns=["female", "male"]
)

# Compute male-female difference
tfidf_df["diff"] = tfidf_df["male"] - tfidf_df["female"]

# Show top nouns
top_male_nouns = tfidf_df.sort_values("diff", ascending=False).head(10)
top_female_nouns = tfidf_df.sort_values("diff", ascending=True).head(10)

In [19]:
top_male_nouns

,female,male,diff
medicine,0.151597,0.165628,0.014031
staff,0.047931,0.059773,0.011843
knowledge,0.114200,0.125307,0.011107
physician,0.063732,0.072574,0.008842
practice,0.024229,0.031521,0.007292
time,0.136849,0.143022,0.006173
anesthesia,0.105677,0.111843,0.006166
year,0.201395,0.207183,0.005789
number,0.028825,0.034538,0.005713
rotation,0.166823,0.172302,0.005479


In [20]:
top_female_nouns

,female,male,diff
health,0.047691,0.032527,-0.015164
research,0.121909,0.108163,-0.013746
identifier,0.356199,0.342753,-0.013446
student,0.314541,0.305929,-0.008612
applicant,0.024851,0.019063,-0.005788
surgery,0.063780,0.058059,-0.005721
education,0.034571,0.029075,-0.005496
community,0.034380,0.029189,-0.005190
leadership,0.032991,0.028001,-0.004990
patient,0.342936,0.338182,-0.004754


In [24]:
all_tokens = (
    top_male_adjs.index.tolist()
    + top_male_verbs.index.tolist()
    + top_male_nouns.index.tolist()
    + top_female_adjs.index.tolist()
    + top_female_verbs.index.tolist()
    + top_female_nouns.index.tolist()
)


all_tokens = list(set(all_tokens))
all_tokens = pd.DataFrame(all_tokens, columns=["token"])
all_tokens = all_tokens.reset_index()
all_tokens = all_tokens.drop('index', axis=1)

In [30]:
all_tokens

,token
0,patient
1,professional
2,spend
3,public
4,staff
5,display
6,clinical
7,applicant
8,build
9,knowledge


In [31]:
# all_tokens.to_csv("../data/top_tokens_tfidf.csv", index=False)